# Setup: `v2_pred_patch` Table for Insert Benchmark

## Schema Description

This notebook documents the existing `v2_pred_patch` table used for the
**Insert 1000 rows into v2_pred_patch** benchmark.

The table already exists and is pre-populated with ~800M rows. We do **not** drop,
truncate, or delete any existing data. The benchmark inserts 1,000 new rows and
then deletes only those 1,000 rows in the cleanup phase.

### Table: `v2_pred_patch`

| Column         | Type            | Key      | Description                                         |
|---------------|-----------------|----------|-----------------------------------------------------|
| id            | SERIAL PK        | Primary  | Auto-incrementing integer identifier                |
| patch_uid     | BIGINT NOT NULL  | Column   | Unique patch identifier (FK-like, no constraint)    |
| embed_coords  | POINT            | Column   | 2D embedding coordinates (x, y)                     |
| grid_cell_i   | INT              | Column   | IJ grid index — i component                         |
| grid_cell_j   | INT              | Column   | IJ grid index — j component                         |
| event_ts      | TIMESTAMPTZ      | Column   | Timestamp of append (DEFAULT now())                 |
| pred_label    | INT              | Column   | Predicted label class id                            |
| patch_coords  | POINT            | Column   | Patch coordinates within source image               |

### Indexes
- `v2_pred_patch_pkey` — B-tree UNIQUE on `id` (PRIMARY KEY)
- `idx_v2_pred_patch_grid_cells` — B-tree on `(grid_cell_i, grid_cell_j)`

### Pre-existing row count
~800,000,200 rows (800M+).

### Setup/Teardown Notes
- **Setup**: No DDL needed — table already exists with data.
- **Benchmark**: Insert 1,000 new rows; record their `id` values.
- **Teardown**: Delete the 1,000 inserted rows by their `id` values.

## Connection
Reads from environment variables `DB_HOST`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`;
falls back to the prototyping defaults if not set.

In [ ]:
import os
import psycopg2

# ---------------------------------------------------------------------------
# Connection parameters — read from env vars, fall back to prototyping defaults
# ---------------------------------------------------------------------------
DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

conn = psycopg2.connect(DSN)
cur = conn.cursor()

# Report PG version and existing table state
cur.execute('SELECT version();')
pg_ver = cur.fetchone()[0].split(',')[0]
print(f'Connected to {pg_ver}')

cur.execute('SELECT COUNT(*) FROM v2_pred_patch;')
count = cur.fetchone()[0]
print(f'Pre-existing row count in v2_pred_patch: {count:,}')

cur.execute("""
    SELECT indexname, indexdef
    FROM pg_indexes
    WHERE tablename = 'v2_pred_patch';
""")
print('\nIndexes on v2_pred_patch:')
for row in cur.fetchall():
    print(f'  {row[0]}: {row[1]}')

cur.execute("""
    SELECT column_name, data_type, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name = 'v2_pred_patch'
    ORDER BY ordinal_position;
""")
print('\nColumn schema:')
for row in cur.fetchall():
    print(f'  {row}')

conn.close()
print('\nSetup verification complete. No DDL changes needed.')

In [ ]:
# ---------------------------------------------------------------------------
# TEARDOWN — Deletes only the 1000 rows inserted by the benchmark
# Run this cell after the benchmark to clean up
# Requires: `inserted_ids` variable from the benchmark notebook
# ---------------------------------------------------------------------------
import os
import psycopg2

DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

# These are the IDs inserted during the benchmark — set them here if running teardown separately
# inserted_ids = [...]   # populate if needed

conn = psycopg2.connect(DSN)
conn.autocommit = False
cur = conn.cursor()

# Count rows before
cur.execute('SELECT COUNT(*) FROM v2_pred_patch;')
before = cur.fetchone()[0]
print(f'Row count before teardown: {before:,}')

# Delete inserted rows using the IDs
cur.execute("""
    DELETE FROM v2_pred_patch
    WHERE id IN (
        SELECT id FROM v2_pred_patch
        WHERE patch_uid >= 999_000_000_000
        LIMIT 2000
    );
""")
deleted = cur.rowcount
conn.commit()

cur.execute('SELECT COUNT(*) FROM v2_pred_patch;')
after = cur.fetchone()[0]
print(f'Rows deleted: {deleted}')
print(f'Row count after teardown: {after:,}')
conn.close()
print('Teardown complete — only benchmark rows removed.')